In [ ]:
import requests
from lxml import html
import os
import re

url = "https://ceo.gujarat.gov.in/Home/ElectionResult"
base_url = "https://ceo.gujarat.gov.in"

xpath = '/html/body/div[1]/div[3]/div/div[2]/main/div[2]/div/div/div[1]/div/div/div/section[2]/div/ul[6]/li/div[2]/div/div/table/tbody//a'

headers = {
    "User-Agent": "Mozilla/5.0"
}

os.makedirs("downloads", exist_ok=True)

def clean_filename(name):
    return re.sub(r'[\\/:*?"<>|]', '', name).strip()

response = requests.get(url, headers=headers)
response.raise_for_status()

tree = html.fromstring(response.content)
links = tree.xpath(xpath)

count = 0

for a in links:
    href = a.get("href")
    name = a.text_content().strip()   # 👈 yahin se naam uth raha h (Abdasa)

    if not href or not name:
        continue

    if href.startswith("/"):
        href = base_url + href

    filename = clean_filename(name) + ".pdf"
    filepath = os.path.join("downloads", filename)

    print("Downloading:", filename)

    try:
        r = requests.get(href, headers=headers)
        r.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(r.content)

        count += 1
        print("Saved ->", filepath)

    except Exception as e:
        print("Failed:", name, e)

print("\nTotal files downloaded:", count)


Downloading: Abdasa.pdf
Saved -> downloads\Abdasa.pdf
Downloading: Mandvi.pdf
Saved -> downloads\Mandvi.pdf
Downloading: Bhuj.pdf


In [1]:
import requests
from lxml import html
import os
import re
from urllib.parse import urlparse

url = "https://ceo.gujarat.gov.in/Home/ElectionResult"
base_url = "https://ceo.gujarat.gov.in"

xpath = '/html/body/div[1]/div[3]/div/div[2]/main/div[2]/div/div/div[1]/div/div/div/section[4]/div/ul[6]/li/div[2]/div/div/table/tbody//a'

headers = {
    "User-Agent": "Mozilla/5.0"
}

# downloads folder
os.makedirs("downloads", exist_ok=True)

def clean_filename(name):
    return re.sub(r'[\\/:*?"<>|]', '', name).strip()

response = requests.get(url, headers=headers)
response.raise_for_status()

tree = html.fromstring(response.content)
links = tree.xpath(xpath)

count = 0
skipped = 0

for a in links:
    href = a.get("href")
    name = a.text_content().strip()   # anchor text se file name

    if not href or not name:
        continue

    # make full URL
    if href.startswith("/"):
        href = base_url + href

    # automatic extension detect from href
    parsed = urlparse(href)
    ext = os.path.splitext(parsed.path)[1]  # ex: '.pdf' or '.xls'
    if not ext:
        ext = ".pdf"  # default if extension not found

    filename = clean_filename(name) + ext
    filepath = os.path.join("downloads", filename)

    # skip if file already exists
    if os.path.exists(filepath):
        print("Skipped (already exists):", filename)
        skipped += 1
        continue

    print("Downloading:", filename)

    try:
        r = requests.get(href, headers=headers)
        r.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(r.content)

        count += 1
        print("Saved ->", filepath)

    except Exception as e:
        print("Failed:", name, e)

print(f"\nTotal files downloaded: {count}")
print(f"Total files skipped (already existed): {skipped}")


Downloading: Abdasa.pdf
Saved -> downloads\Abdasa.pdf
Downloading: Mandvi.pdf
Saved -> downloads\Mandvi.pdf
Downloading: Bhuj.pdf
Saved -> downloads\Bhuj.pdf
Downloading: Anjar.pdf
Saved -> downloads\Anjar.pdf
Downloading: Gandhidham (SC).pdf
Saved -> downloads\Gandhidham (SC).pdf
Downloading: Rapar.pdf
Saved -> downloads\Rapar.pdf
Downloading: Morbi.pdf
Saved -> downloads\Morbi.pdf
Downloading: Vav.pdf
Saved -> downloads\Vav.pdf
Downloading: Tharad.pdf
Saved -> downloads\Tharad.pdf
Downloading: Dhanera.pdf
Saved -> downloads\Dhanera.pdf
Downloading: Danta (ST).pdf
Saved -> downloads\Danta (ST).pdf
Downloading: Palanpur.pdf
Saved -> downloads\Palanpur.pdf
Downloading: Deesa.pdf
Saved -> downloads\Deesa.pdf
Downloading: Deodar.pdf
Saved -> downloads\Deodar.pdf
Downloading: Vadgam (SC).pdf
Saved -> downloads\Vadgam (SC).pdf
Downloading: Kankrej.pdf
Saved -> downloads\Kankrej.pdf
Downloading: Radhanpur.pdf
Saved -> downloads\Radhanpur.pdf
Downloading: Chanasma.pdf
Saved -> downloads\Chana

KeyboardInterrupt: 

In [2]:
import requests
from lxml import html
import os
import re

# URL aur base URL
url = "https://ceo.gujarat.gov.in/Home/ElectionResult"
base_url = "https://ceo.gujarat.gov.in"

# XPath jahan se files ke links nikalne hain
xpath = '/html/body/div[1]/div[3]/div/div[2]/main/div[2]/div/div/div[1]/div/div/div/section[2]/div/ul[6]/li/div[2]/div/div/table/tbody//a'

# Folder jahan files save hongi
download_folder = "downloads"
os.makedirs(download_folder, exist_ok=True)

# Webpage request
response = requests.get(url)
response.raise_for_status()  # agar page load nahi hua toh error dega

# Parse HTML
tree = html.fromstring(response.content)

# Sari links nikal lo
links = tree.xpath(xpath + '/@href')

# Agar relative URL hai toh base_url add karo
links = [link if link.startswith('http') else base_url + link for link in links]

print(f"Total {len(links)} links found.")

# Files download karna
for idx, link in enumerate(links, 1):
    # File extension detect karo
    ext_match = re.search(r'\.(pdf|xls|xlsx)$', link, re.IGNORECASE)
    ext = ext_match.group(0) if ext_match else '.pdf'  # default .pdf

    # File name
    file_name = f"{idx}{ext}"
    file_path = os.path.join(download_folder, file_name)

    # Agar file already exist karti hai toh skip karo
    if os.path.exists(file_path):
        print(f"Skipping {file_name}, already downloaded.")
        continue

    # File download
    try:
        print(f"Downloading {file_name} from {link}...")
        file_data = requests.get(link)
        file_data.raise_for_status()
        with open(file_path, 'wb') as f:
            f.write(file_data.content)
        print(f"{file_name} downloaded successfully.")
    except Exception as e:
        print(f"Failed to download {link}: {e}")

print("All done!")


Total 364 links found.
1.pdf downloaded successfully.


KeyboardInterrupt: 